In [4]:
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
    print("✅ Drive mounted successfully")
else:
    print("✅ Drive already mounted, skipping...")

✅ Drive already mounted, skipping...


In [5]:
#configuration notebook

import os
os.chdir("/content/drive/My Drive/Colab Notebooks")
%run oo_config.ipynb

In [6]:
import os
import requests
import json
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField,
    StringType, BooleanType, DoubleType
)

# --- Config ---
load_dotenv()
ACCESS_KEY = G_AviationStackAPI
BASE_URL = 'https://api.aviationstack.com/v1/flights'

# --- Spark Session ---
spark = SparkSession.builder \
    .appName("AviationFlights") \
    .getOrCreate()

# --- Fetch API ---
params = {'access_key': ACCESS_KEY}
api_result = requests.get(BASE_URL, params)
api_response = api_result.json()

flights = api_response.get('data', [])
# --- Convert raw JSON directly to Spark DataFrame ---
json_rdd = spark.sparkContext.parallelize([json.dumps(f) for f in flights])
df = spark.read.json(json_rdd)

df = df.filter(~col("flight_status").isin(["scheduled","landed"]))
# --- Show ---
df.printSchema()
df.show(truncate=False)


root
 |-- aircraft: struct (nullable = true)
 |    |-- iata: string (nullable = true)
 |    |-- icao: string (nullable = true)
 |    |-- icao24: string (nullable = true)
 |    |-- registration: string (nullable = true)
 |-- airline: struct (nullable = true)
 |    |-- iata: string (nullable = true)
 |    |-- icao: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- arrival: struct (nullable = true)
 |    |-- actual: string (nullable = true)
 |    |-- actual_runway: string (nullable = true)
 |    |-- airport: string (nullable = true)
 |    |-- baggage: string (nullable = true)
 |    |-- delay: long (nullable = true)
 |    |-- estimated: string (nullable = true)
 |    |-- estimated_runway: string (nullable = true)
 |    |-- gate: string (nullable = true)
 |    |-- iata: string (nullable = true)
 |    |-- icao: string (nullable = true)
 |    |-- scheduled: string (nullable = true)
 |    |-- terminal: string (nullable = true)
 |    |-- timezone: string (nullable = true)
 